# Jansen-Rit SBI Pipeline

Interactive pipeline for Stats-Only SBI parameter inference on the Jansen-Rit neural mass model.

- **`config.py`** — all hyperparameters/priors
- **`simulator.py`** — Jansen-Rit physics, observation noise, Sobol sampling
- **`forward_model.py`** — lead field model (EEG sensor-space projection)
- **`embedding.py`** — statistical feature extraction + gating
- **`metrics.py`** — R²/NRMSE/Pearson and posterior evaluation
- **`training.py`** — training orchestration and ablation analysis
- **`run.py`** — CLI entrypoint (this notebook is the interactive alternative)

Three independently-toggled experiment phases (section 5): **baseline** training, **gold-standard
ablation**, and **gate activation comparison**.

---
## 1. Setup and Imports

In [ ]:
%load_ext autoreload
%autoreload 2

import time
import warnings
import numpy as np
import torch
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore', message='.*triangular_solve.*')
warnings.filterwarnings('ignore', message='.*Torchinductor does not support.*')
warnings.filterwarnings('ignore', message='.*TensorFloat32.*')
warnings.filterwarnings('ignore', message='.*torch._dynamo.*')
warnings.filterwarnings('ignore', message='.*Moving.*to the data_device.*')

if torch.cuda.is_available():
    torch.set_float32_matmul_precision('high')
    print(f"CUDA available: {torch.cuda.get_device_name(0)}")
else:
    print("CUDA not available, using CPU")

In [ ]:
from config import Config
from simulator import simulate_jansen_rit, generate_sobol_parameters
from embedding import StatsOnlyEmbeddingNet
from training import train_and_evaluate, run_retrain_ablation_analysis
from metrics import compute_r2, compute_nrmse
from forward_model import LeadFieldModel, clear_cache
from experiment_log import log_run, log_gating_importance, log_ablation, log_posterior_correlation
from post_analysis import compute_posterior_correlation

print("Modules imported")

---
## 2. Configuration

In [ ]:
config = Config()

print(f"Simulations:    {config.NUM_SIMULATIONS}")
print(f"Eval samples:   {config.N_EVAL_SAMPLES}")
print(f"Max epochs:     {config.MAX_EPOCHS} (early stop after {config.STOP_AFTER_EPOCHS})")
print(f"Device:         {'CUDA' if torch.cuda.is_available() else 'CPU'}")
print(f"Forward model:  {config.SOURCE_LOCATION if config.USE_FORWARD_MODEL else 'DISABLED'}")
print(f"Channels:       {config.N_CHANNELS}")
print(f"Parameters:     {config.PARAM_NAMES}")

---
## 3. Forward Model

In [ ]:
if config.USE_FORWARD_MODEL:
    lead_field_model = LeadFieldModel(config.SOURCE_LOCATION, verbose=True)
    channel_names = lead_field_model.resolve_channels(config) if config.N_CHANNELS > 1 else None
    print(f"Electrode(s): {channel_names or lead_field_model.selected_electrode}")
else:
    lead_field_model = None
    channel_names = None

---
## 4. Training Data

In [ ]:
torch.manual_seed(config.SEED)
np.random.seed(config.SEED)

theta_train = generate_sobol_parameters(
    config.NUM_SIMULATIONS, config.PRIOR_MIN, config.PRIOR_MAX, seed=config.SEED
)
x_train = simulate_jansen_rit(
    theta_train, config=config, use_gpu=torch.cuda.is_available(), lead_field_model=lead_field_model
)

print(f"theta_train: {theta_train.shape}   x_train: {x_train.shape}")

---
## 4b. Fixed Evaluation Set

One held-out test set, sampled once with its own seed, reused across every phase below (baseline,
ablation, gating). Without this, each run/condition would draw its own random test points, which
mixes eval-sampling noise into any comparison between them.

In [ ]:
from sbi.utils import BoxUniform

EVAL_SEED = 999  # separate from the training seeds
torch.manual_seed(EVAL_SEED)
np.random.seed(EVAL_SEED)

eval_prior = BoxUniform(low=torch.tensor(config.PRIOR_MIN), high=torch.tensor(config.PRIOR_MAX))
theta_test_fixed = eval_prior.sample((config.N_EVAL_SAMPLES,))
x_test_fixed = simulate_jansen_rit(
    theta_test_fixed.cpu(), config=config, use_gpu=torch.cuda.is_available(),
    lead_field_model=lead_field_model, fixed_snr_db=config.EVAL_SNR_DB
)

print(f"Fixed eval set: {config.N_EVAL_SAMPLES} samples (seed={EVAL_SEED})")

---
## 5. Experiment Phases

Three independent phases, each with its own on/off toggle. Run any subset — ablation needs
`results_by_seed` (from Baseline) to compute R² drops against; gating is fully independent.

In [ ]:
SEEDS = [0, 1, 10, 99, 100]   #[5, 15, 50, 55, 105]
RUN_NAME = "default"
NOTES = ""
 
RUN_BASELINE = True
RUN_ABLATION = True  # Ablation study: retrain with each parameter removed
RUN_GATING = False  # Channel-tied gate, sigmoid vs. softmax

---
### 5a. Baseline Training

In [ ]:
if RUN_BASELINE:
    config.USE_GATING = False
    results_by_seed = {}
    run_labels_by_seed = {}
    t0 = time.time()

    for seed in SEEDS:
        print(f"\n{'='*70}\n  BASELINE — SEED {seed}\n{'='*70}")
        config.SEED = seed
        seed_results = train_and_evaluate(
            config=config, theta_train=theta_train, x_train=x_train, seed=seed,
            verbose=True, theta_test=theta_test_fixed, x_test=x_test_fixed,
        )
        results_by_seed[seed] = seed_results
        run_label = log_run(config, seed_results, run_name=RUN_NAME, notes=f"{NOTES} seed={seed}")
        run_labels_by_seed[seed] = run_label

        mean_corr = compute_posterior_correlation(
            seed_results.posterior, x_test_fixed, config.PARAM_NAMES, verbose=False
        )
        log_posterior_correlation(config, seed, config.PARAM_NAMES, mean_corr,
                                   run_label=run_label, notes=f"{NOTES} seed={seed}")
        print(f"Logged run: {run_label}")

    results = results_by_seed[SEEDS[-1]]
    run_label = run_labels_by_seed[SEEDS[-1]]
    print(f"\nBaseline complete in {(time.time()-t0)/60:.1f} min")
else:
    print("RUN_BASELINE=False — skipped (expects results_by_seed already in memory).")

In [ ]:
if 'results_by_seed' not in globals() or not results_by_seed:
    print("No baseline results yet.")
else:
    _, _, param_names = config.get_effective_prior_bounds()
    seed_list = [results_by_seed[s] for s in SEEDS]

    def _agg(attr, idx=None):
        vals = np.array([getattr(r, attr)[idx] if idx is not None else getattr(r, attr) for r in seed_list])
        return vals.mean(), vals.std()

    r2_m, r2_s = _agg('mean_r2')
    print(f"Mean R² across {len(SEEDS)} seeds: {r2_m:.4f} ± {r2_s:.4f}\n")
    print(f"{'Parameter':<12} {'R²':>16} {'NRMSE':>16} {'Cov90':>16}")
    for i, name in enumerate(param_names):
        r2i_m, r2i_s = _agg('r2_per_param', i)
        nr_m, nr_s = _agg('nrmse_per_param', i)
        cov_m, cov_s = _agg('coverage_90', i)
        print(f"{name:<12} {r2i_m:6.3f}±{r2i_s:<6.3f}   {nr_m:6.3f}±{nr_s:<6.3f}   {cov_m:6.2f}±{cov_s:<6.2f}")

In [ ]:
if 'results' in globals():
    _, _, param_names = config.get_effective_prior_bounds()
    n_params = len(param_names)
    fig, axes = plt.subplots(1, n_params, figsize=(4.5 * n_params, 3.5))
    for i, ax in enumerate(axes):
        ax.scatter(results.theta_true[:, i], results.theta_pred[:, i], alpha=0.5, s=20, c='steelblue')
        lims = [min(results.theta_true[:, i].min(), results.theta_pred[:, i].min()),
                max(results.theta_true[:, i].max(), results.theta_pred[:, i].max())]
        ax.plot(lims, lims, 'r--', linewidth=2)
        ax.set_xlabel('True'); ax.set_ylabel('Predicted')
        ax.set_title(f'{param_names[i]}  R²={results.r2_per_param[i]:.3f}')
        ax.set_aspect('equal', 'box')
    plt.suptitle(f'Parameter Recovery (seed={SEEDS[-1]})', y=1.02)
    plt.tight_layout()
    plt.show()

---
### 5b. Gold-Standard Ablation

Retrains once per feature, excluding it, and measures the R² drop against baseline — the causal
ground truth for feature importance. Requires `RUN_BASELINE` to have populated `results_by_seed`.

In [ ]:
if RUN_ABLATION:
    if 'results_by_seed' not in globals() or not results_by_seed:
        print("Ablation needs baseline results first — set RUN_BASELINE=True and run that cell.")
    else:
        ablation_results_by_seed = {}
        t0 = time.time()

        for seed in SEEDS:
            print(f"\n{'='*70}\n  ABLATION — SEED {seed}\n{'='*70}")
            ablation_results = run_retrain_ablation_analysis(
                theta_train=theta_train, x_train=x_train, config=config,
                baseline_r2=results_by_seed[seed].r2_per_param, seed=seed, verbose=True,
                lead_field_model=lead_field_model, channel_names=channel_names,
                theta_test=theta_test_fixed, x_test=x_test_fixed,
            )
            ablation_results_by_seed[seed] = ablation_results
            log_ablation(config, ablation_results, baseline_run_label=run_labels_by_seed[seed], notes=f"{NOTES} seed={seed}")

        print(f"\nAblation complete in {(time.time()-t0)/60:.1f} min")
else:
    print("RUN_ABLATION=False — skipped.")

In [ ]:
if 'ablation_results_by_seed' not in globals() or not ablation_results_by_seed:
    print("No ablation results yet.")
else:
    feat_names = list(next(iter(ablation_results_by_seed.values())).keys())
    mean_drops = [np.mean([ablation_results_by_seed[s][f]['mean_r2_drop'] for s in SEEDS]) for f in feat_names]
    std_drops = [np.std([ablation_results_by_seed[s][f]['mean_r2_drop'] for s in SEEDS]) for f in feat_names]

    order = np.argsort(mean_drops)[::-1]
    feat_sorted = [feat_names[i] for i in order]
    mean_sorted = [mean_drops[i] for i in order]
    std_sorted = [std_drops[i] for i in order]

    print("R² drop when feature removed (mean ± std across seeds), sorted by importance:")
    for f, m, s in zip(feat_sorted, mean_sorted, std_sorted):
        print(f"  {f:20s} {m:+.4f} ± {s:.4f}")

    plt.figure(figsize=(10, 6))
    colors = ['#d62728' if x > 0 else '#2ca02c' for x in mean_sorted]
    plt.barh(feat_sorted, mean_sorted, xerr=std_sorted, color=colors, capsize=3)
    plt.axvline(x=0, color='black', linewidth=0.5)
    plt.xlabel('R² Drop when Feature Removed')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

---
### 5c. Gate Activation Comparison

The gate learns one weight per base feature, shared identically across every channel (pooled
across channels before the gate net, tiled back out after) — matching ablation's granularity
directly, so gate weights compare against it with no averaging step needed.

- **`sigmoid`**: independent gating, no competition between features.
- **`softmax`**: weights forced to sum to 1 across the 7 features — competitive attention.

Comparing the two isolates the effect of the normalization mechanism alone.

In [ ]:
if RUN_GATING:
    config.USE_GATING = True
    gate_results = {"sigmoid": {}, "softmax": {}}
    t0 = time.time()

    for activation in gate_results:
        config.GATE_ACTIVATION = activation
        for seed in SEEDS:
            print(f"\n{'='*70}\n  GATING {activation} — SEED {seed}\n{'='*70}")
            config.SEED = seed
            seed_results = train_and_evaluate(
                config=config, theta_train=theta_train, x_train=x_train, seed=seed,
                verbose=True, theta_test=theta_test_fixed, x_test=x_test_fixed,
            )
            gate_results[activation][seed] = seed_results
            run_label = log_run(config, seed_results, run_name=f"gating_{activation}", notes=f"{NOTES} seed={seed}")
            log_gating_importance(config, seed_results.gate_importance, run_label=run_label, notes=f"{NOTES} seed={seed}")

            mean_corr = compute_posterior_correlation(
                seed_results.posterior, x_test_fixed, config.PARAM_NAMES, verbose=False
            )
            log_posterior_correlation(config, seed, config.PARAM_NAMES, mean_corr,
                                       run_label=run_label, notes=f"{NOTES} seed={seed} activation={activation}")

    config.USE_GATING = False
    print(f"\nGating comparison complete in {(time.time()-t0)/60:.1f} min")
else:
    print("RUN_GATING=False — skipped.")

In [ ]:
if 'gate_results' not in globals():
    print("No gating results yet.")
else:
    base_feature_names = [f for f in StatsOnlyEmbeddingNet.ALL_FEATURE_NAMES if f not in config.EXCLUDED_FEATURES]
    # Gate weights are channel-tied (identical across channels for a given base
    # feature), so any one channel is representative when N_CHANNELS>1. In
    # single-channel mode feature_names() returns unprefixed names, so the key
    # is just the feature name itself, not "{channel}_{feature}".
    def _gate_weight_key(feat):
        return feat if config.N_CHANNELS == 1 else f"{channel_names[0]}_{feat}"

    print("Mean R² by gate activation:")
    for activation, per_seed in gate_results.items():
        r2s = [per_seed[s].mean_r2 for s in SEEDS]
        print(f"  {activation:8s} {np.mean(r2s):.4f} ± {np.std(r2s):.4f}")

    gate_weights = {}
    for activation, per_seed in gate_results.items():
        gate_weights[activation] = {
            feat: np.mean([per_seed[s].gate_importance['mean_weights'][_gate_weight_key(feat)] for s in SEEDS])
            for feat in base_feature_names
        }

    print(f"\n{'Feature':<20} {'sigmoid':>12} {'softmax':>12}")
    for feat in base_feature_names:
        print(f"{feat:<20} {gate_weights['sigmoid'][feat]:>12.4f} {gate_weights['softmax'][feat]:>12.4f}")

In [ ]:
from scipy.stats import spearmanr, kendalltau

if 'gate_weights' not in globals():
    print("Run gate training first.")
elif 'ablation_results_by_seed' not in globals() or not ablation_results_by_seed:
    print("Needs ablation ground truth — set RUN_ABLATION=True and run that cell first.")
else:
    ablation_drop = {f: np.mean([ablation_results_by_seed[s][f]['mean_r2_drop'] for s in SEEDS]) for f in base_feature_names}
    ablation_vals = [ablation_drop[f] for f in base_feature_names]

    ranks = {}
    for activation in ["sigmoid", "softmax"]:
        gate_vals = [gate_weights[activation][f] for f in base_feature_names]
        rho, p = spearmanr(ablation_vals, gate_vals)
        tau, pt = kendalltau(ablation_vals, gate_vals)
        print(f"{activation:8s} Spearman ρ={rho:+.3f} (p={p:.3f})   Kendall τ={tau:+.3f} (p={pt:.3f})")
        ranks[activation] = [base_feature_names[i] for i in np.argsort(gate_vals)[::-1]]

    print(f"\n{'Rank':<6} {'sigmoid':<20} {'softmax':<20}")
    for i in range(len(base_feature_names)):
        print(f"{i+1:<6} {ranks['sigmoid'][i]:<20} {ranks['softmax'][i]:<20}")

    if ranks['sigmoid'][0] == ranks['softmax'][0]:
        print(f"\nBoth activations agree on the #1 feature ({ranks['sigmoid'][0]}).")
    else:
        print(f"\nTop feature differs: sigmoid picks '{ranks['sigmoid'][0]}', softmax picks '{ranks['softmax'][0]}'.")

---
## 6. Cleanup

In [ ]:
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("GPU memory cleared")